# Stage 10: Answering with citations, citation checks and judging

For each of the 40 eval questions:
1. **Retrieve** the top 10 excerpts (dense + BM25, fused, reranked by MiniLM).
2. **Answer** with Claude Sonnet 5.5, which may call `get_image` for images the excerpts mention. It must answer
   only from the excerpts, say plainly when they don't cover something, and cite an exact quote, excerpt id and page
   for every claim (`src/handbook_qa/answer.py`).
3. **Check every citation automatically**: is the quote really in the cited excerpt, and on the cited page?
   (`src/handbook_qa/citations.py`)
4. **Judge** each answer against the reference answer with Claude Opus 5.5 (`src/handbook_qa/judge.py`).

**How the current answers and judgements were produced:** no API key is connected yet, so they come from Claude
Code subagents on the same models (Sonnet for answering, Opus for judging), each given the exact system prompt and
user message the API call would send. They are saved in `artifacts/answers/`. With a key, `RUN_API = True`
produces fresh ones through the API (key from ANTHROPIC_API_KEY).

In [1]:
import collections
import json
from dataclasses import asdict

import pandas as pd

from handbook_qa import evaluation as ev
from handbook_qa.answer import (
    AnswerResult,
    Citation,
    HandbookAnswer,
    answer_question,
    build_user_message,
)
from handbook_qa.citations import PageIndex, check_citations
from handbook_qa.config import ROOT, load_settings
from handbook_qa.judge import SCORES, Judgement, judge_one
from handbook_qa.pdf_extract import load_articles
from handbook_qa.retrieval import Retriever
from handbook_qa.tools import ImageStore

RUN_API = False          # answer and judge through the Claude API (needs a key)
settings = load_settings()
ANSWERS = ROOT / "artifacts" / "answers" / "answers.jsonl"
JUDGEMENTS = ROOT / "artifacts" / "answers" / "judgements.jsonl"
pd.set_option("display.max_colwidth", 120)
print("answer model:", settings.answer.model, "| judge model:", settings.judge.model,
      "| excerpts per question:", settings.answer.top_k, "| reranker:", settings.retrieval.rerank_model)

# The eval set later grew to 100 questions; the 60 held-out ones are kept for the final face-off (notebook 12).
questions = [q for q in ev.load_eval_set(ROOT / "evals" / "handbook_eval.jsonl") if q.split == "dev"]
retriever = Retriever.from_settings(settings)
index = PageIndex(load_articles(settings.paths.articles),
                  [json.loads(l) for l in settings.paths.image_index.read_text(encoding="utf-8").splitlines()])
images = ImageStore(settings.paths.image_index, ROOT)
print(ev.resolve(questions, retriever.items) or "answer key resolves")

answer model: claude-sonnet-5-5 | judge model: claude-opus-5-5 | excerpts per question: 10 | reranker: Xenova/ms-marco-MiniLM-L-6-v2


C:\Users\shaba\Documents\extra project\paradigm_assessment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


answer key resolves


## 1. Retrieve

In [2]:
hits = {q.id: retriever.search(q.question, k=settings.answer.top_k) for q in questions}
recall = pd.Series({q.id: ev.chunk_recall([h.item.id for h in hits[q.id]], q, settings.answer.top_k) for q in questions})
print(f"chunk recall@{settings.answer.top_k} of what the model is given: {recall.mean():.3f}; "
      f"questions with every expected chunk: {(recall == 1).mean():.0%}")

chunk recall@10 of what the model is given: 0.846; questions with every expected chunk: 75%


What the model sees, for one question (the excerpts carry `[page N]` markers so it can cite pages):

In [3]:
message = build_user_message(questions[0].question, hits[questions[0].id], index)
print(message[:1800], "\n...\n", message[-300:])

Handbook excerpts, most relevant first:

<excerpt id="img-p0041-x258" kind="image" source="contents/handbook/company/brand-assets.md" pages="41">
[page 41]
Communication tier list ranking PostHog communication methods from S tier to F tier.
A tier list titled Communication tier list. S tier: Pull requests. A tier: Request for comments and GitHub issues. B tier: Public Slack channels. C tier: Carrier pigeon and Chemtrails. D tier: Private messages and Fax machine. E tier: Email. F tier: Tapping someone on the shoulder and saying, "You got a minute?". Each tier is a colored label with an icon. Some entries are jokes (carrier pigeon, chemtrails, fax machine). It ranks public, written, asynchronous channels highest and interruptive ones lowest.
Text in the image: COMMUNICATION TIER LIST S TIER PULL REQUESTS A TIER REQUEST FOR COMMENTS GITHUB ISSUES B TIER PUBLIC SLACK CHANNELS C TIER CARRIER PIGEON CHEMTRAILS D TIER PRIVATE MESSAGES FAX MACHINE E TIER EMAIL F TIER TAPPING SOMEONE ON THE SH

## 2. Answer

In [4]:
# Answer through the API when enabled
if RUN_API:
    from handbook_qa.llm import client as make_client
    client = make_client()
    results = {q.id: answer_question(client, settings.answer, q.question, hits[q.id], index, images)
               for q in questions}
    ANSWERS.parent.mkdir(parents=True, exist_ok=True)
    with ANSWERS.open("w", encoding="utf-8") as f:
        for q in questions:
            r = results[q.id]
            f.write(json.dumps({"id": q.id, "source": r.source, "excerpt_ids": r.excerpt_ids,
                                "images_viewed": r.images_viewed, **r.answer.model_dump()}, ensure_ascii=False) + "\n")

# Load saved answers and check their citations
saved = {row["id"]: row for row in map(json.loads, ANSWERS.read_text(encoding="utf-8").splitlines())}
results = {}
for q in questions:
    row = saved[q.id]
    # The saved answer was made from these excerpts; retrieval here must reproduce them for the checks to be fair.
    assert row["excerpt_ids"] == [h.item.id for h in hits[q.id]], f"{q.id}: retrieval changed since answering"
    answer = HandbookAnswer.model_validate({k: row[k] for k in ("coverage", "answer", "citations")})
    excerpts = {h.item.id: h.item for h in hits[q.id]}
    results[q.id] = AnswerResult(q.question, answer, row["excerpt_ids"], row["images_viewed"],
                                 check_citations(answer.citations, excerpts, index), source=row["source"])
print(len(results), "answers; sources:", dict(collections.Counter(r.source for r in results.values())))
print("retrieval reproduced the saved excerpts for every question")

40 answers; sources: {'claude-code-subagent:sonnet (stand-in for claude-sonnet-5-5 via API)': 40}
retrieval reproduced the saved excerpts for every question


In [5]:
coverage = pd.Series({qid: r.answer.coverage for qid, r in results.items()})
print("coverage labels:", coverage.value_counts().to_dict())
print("images fetched:", {qid: r.images_viewed for qid, r in results.items() if r.images_viewed})

coverage labels: {'full': 34, 'partial': 5, 'not_covered': 1}
images fetched: {}


Two answers in full:

In [6]:
for qid in ["C1-01", "A-11"]:
    r = results[qid]
    print(f"=== {qid}: {r.question}\ncoverage: {r.answer.coverage}\n\n{r.answer.answer}\n")
    for c in r.answer.citations:
        print(f'  [{r.answer.citations.index(c) + 1}] {c.source_id}, p{c.page}: "{c.quote}"')
    print()

=== C1-01: If I start on July 21st, how is my commission calculated during ramp?
coverage: full

Your first three months of commission are paid at 100% fixed OTE, calculated from your start date [1]. Because July 21st is after the 15th, the example for a mid-month start applies: a Jan 17th start gets two months of fixed OTE in the first quarter and one in the next, then two months of quota'ed commission [2]. The handbook's timeline for a 7/21/2025 hire shows the same: Q3 is 1 month not employed plus 2 months at 100% fixed OTE, and Q4 is 1 month at 100% fixed OTE plus 2 months of 'your attainment' OTE [3].

So: August and September (Q3) and October (Q4) are fixed at 100% OTE, and from November your commission is based on your quota attainment.

  [1] growth/sales/new-hire-onboarding#06, p616: "Your first three months of commission are paid at 100% fixed OTE. This will be calculated based on the date you start."
  [2] growth/sales/new-hire-onboarding#06, p616: "If you start on Jan 17th, 

## 3. Citation checks

Every quote is looked up in the excerpt it cites (exact text, ignoring only whitespace and curly-vs-straight
quotes) and its page is looked up in the extracted PDF.

In [7]:
checks = pd.DataFrame([{"id": qid, **{k: v for k, v in asdict(c).items()}, "ok": c.ok}
                       for qid, r in results.items() for c in r.checks])
print(f"{len(checks)} citations in {checks['id'].nunique()} answers")
print(f"quote found in the cited excerpt: {checks['in_cited_excerpt'].mean():.1%}")
print(f"quote found in some excerpt given to the model: {checks['in_any_excerpt'].mean():.1%}")
print(f"page number correct: {checks['page_ok'].mean():.1%}")
print(f"citation fully correct (right excerpt and right page): {checks['ok'].mean():.1%}")
per_answer = checks.groupby("id")["ok"].all()
print(f"answers whose citations are all correct: {per_answer.mean():.0%}")

196 citations in 39 answers
quote found in the cited excerpt: 100.0%
quote found in some excerpt given to the model: 100.0%
page number correct: 100.0%
citation fully correct (right excerpt and right page): 100.0%
answers whose citations are all correct: 100%


In [8]:
problems = checks[~checks["ok"]][["id", "number", "source_id", "page", "actual_pages", "in_cited_excerpt",
                                  "in_any_excerpt", "quote"]]
print(len(problems), "citations with a problem")
problems

0 citations with a problem


,id,number,source_id,page,actual_pages,in_cited_excerpt,in_any_excerpt,quote


A clean result is only meaningful if the check would catch bad citations. Negative control: corrupt every real
citation in three ways and count how many the check flags.

In [9]:
flagged = collections.Counter()
for qid, r in results.items():
    excerpts = {h.item.id: h.item for h in hits[qid]}
    for c in r.answer.citations:
        flagged["citations"] += 1
        words = c.quote.split()
        if len(words) > 2:
            words[len(words) // 2] = "zebra"
            flagged["word changed: flagged"] += not check_citations(
                [Citation(source_id=c.source_id, page=c.page, quote=" ".join(words))], excerpts, index)[0].ok
        else:
            flagged["word changed: too short to change"] += 1
        flagged["page + 1: flagged"] += not check_citations(
            [Citation(source_id=c.source_id, page=c.page + 1, quote=c.quote)], excerpts, index)[0].ok
        other = next((i for i in excerpts if i != c.source_id and c.quote.lower() not in excerpts[i].text.lower()), None)
        if other:
            flagged["wrong excerpt id: flagged"] += not check_citations(
                [Citation(source_id=other, page=c.page, quote=c.quote)], excerpts, index)[0].in_cited_excerpt
dict(flagged)

{'citations': 196,
 'word changed: flagged': 195,
 'page + 1: flagged': 196,
 'wrong excerpt id: flagged': 196,
 'word changed: too short to change': 1}

## 4. Judging

Opus grades each answer on correctness, completeness, groundedness and coverage honesty, and says whether the
excerpts contained the evidence at all (separating retrieval misses from answering mistakes).

In [10]:
# Judge through the API when enabled
if RUN_API:
    from handbook_qa.llm import client as make_client
    client = make_client()
    with JUDGEMENTS.open("w", encoding="utf-8") as f:
        for q in questions:
            message = build_user_message(q.question, hits[q.id], index)
            verdict = judge_one(client, settings.judge, q, results[q.id], message)
            f.write(json.dumps({"id": q.id, "source": f"anthropic-api:{settings.judge.model}",
                                **verdict.model_dump()}, ensure_ascii=False) + "\n")

# Load judgements and tally verdicts
judged = {row["id"]: row for row in map(json.loads, JUDGEMENTS.read_text(encoding="utf-8").splitlines())}
verdicts = pd.DataFrame([{"id": q.id, "type": q.type, **{k: judged[q.id][k] for k in Judgement.model_fields}}
                         for q in questions])
print("judge source:", dict(collections.Counter(j["source"] for j in judged.values())))
for dim in ["correctness", "completeness", "groundedness", "coverage_honesty", "evidence_in_excerpts"]:
    print(f"{dim:21}", verdicts[dim].value_counts().to_dict())

judge source: {'claude-code-subagent:opus (stand-in for claude-opus-5-5 via API)': 40}
correctness           {'correct': 39, 'partially_correct': 1}
completeness          {'complete': 30, 'partial': 7, 'missing': 3}
groundedness          {'grounded': 38, 'minor_unsupported': 2}
coverage_honesty      {'appropriate': 39, 'overclaimed': 1}
evidence_in_excerpts  {'all': 30, 'some': 7, 'none': 3}


In [11]:
scores = pd.DataFrame({dim: verdicts[dim].map(values) for dim, values in SCORES.items()})
scores["type"] = verdicts["type"]
summary = scores.groupby("type").mean().round(2)
summary.loc["all"] = scores.drop(columns="type").mean().round(2)
summary.insert(0, "questions", list(verdicts.groupby("type").size()) + [len(verdicts)])
summary

,questions,correctness,completeness,groundedness,coverage_honesty
type,,,,,
existing,11,1.00,0.95,0.91,1.00
multi-article,12,0.96,0.75,1.00,0.92
multi-section,8,1.00,0.94,1.00,1.00
reworded,9,1.00,0.72,1.00,1.00
all,40,0.99,0.84,0.98,0.98


Is a wrong or incomplete answer a retrieval problem or an answering problem?

In [12]:
pd.crosstab(verdicts["evidence_in_excerpts"], verdicts["correctness"], margins=True)

correctness,correct,partially_correct,All
evidence_in_excerpts,,,
all,30,0,30
none,2,1,3
some,7,0,7
All,39,1,40


In [13]:
pd.crosstab(verdicts["evidence_in_excerpts"], verdicts["completeness"], margins=True)

completeness,complete,missing,partial,All
evidence_in_excerpts,,,,
all,27,0,3,30
none,0,3,0,3
some,3,0,4,7
All,30,3,7,40


Answers the judge didn't rate fully correct, complete, grounded and honest:

In [14]:
flagged = verdicts[(verdicts["correctness"] != "correct") | (verdicts["completeness"] != "complete")
                   | (verdicts["groundedness"] != "grounded") | (verdicts["coverage_honesty"] != "appropriate")]
flagged.assign(question=[results[i].question for i in flagged["id"]])[
    ["id", "type", "question", "correctness", "completeness", "groundedness", "coverage_honesty",
     "evidence_in_excerpts", "rationale"]]

,id,type,question,correctness,completeness,groundedness,coverage_honesty,evidence_in_excerpts,rationale
0,A-01,existing,Which communication channels are best and worst at PostHog?,correct,complete,minor_unsupported,appropriate,some,"The answer reproduces the tier list correctly: pull requests (S), RFCs and GitHub issues (A), public Slack (B), emai..."
11,A-12,multi-article,"I'm thinking about quitting. How much notice do I give, who do I tell, and do I have to send back my work laptop?",correct,partial,grounded,appropriate,some,"Notice and who to tell are answered correctly: 30 days by default and worked through, email people@posthog.com, and ..."
14,B-03,existing,How much RAM does the 4-core codespace have?,correct,complete,minor_unsupported,appropriate,all,"The answer correctly gives 16GB of RAM for the 4-core machine type, citing the screenshot description of the machine..."
16,B-05,reworded,A customer accidentally sent sensitive personal info in some of their event properties. Can we scrub just that data ...,correct,partial,grounded,appropriate,all,"The points the answer makes are right: stop sending the data first (client-side redaction or a CDP transformation), ..."
21,B-10,multi-article,"If something breaks badly on a Saturday, will anyone from the support team be watching the incident, and who handles...",correct,partial,grounded,appropriate,some,"What the answer says matches the reference: support has no on-call coverage, watches incidents only in business hour..."
23,C1-02,existing,How do we handle refund requests?,correct,partial,grounded,appropriate,all,"Everything the answer states is accurate and quoted from the refunds excerpts: the 30-day window, no fraud or abuse,..."
25,C1-04,reworded,A big customer asked whether they can pay for PostHog through their Amazon cloud account instead of us invoicing the...,correct,missing,grounded,appropriate,none,"None of the excerpts mention AWS Marketplace or paying through a cloud provider, so the assistant's not_covered answ..."
29,C1-08,multi-article,A startup's free credits from our startup program are about to run out. What can I offer them to keep them on PostHog?,correct,partial,grounded,appropriate,some,The answer correctly gives the startup roll-off offer from the reference: two months of free credit on a prepaid dea...
30,C1-09,multi-article,I want to let one of my paying customers try another PostHog product for free for a few weeks before they commit. Ho...,partially_correct,missing,grounded,overclaimed,none,"The reference sets up a single-product trial for a paying customer with a billing limit (about 30 days, 60 max) or o..."
34,C2-04,reworded,Do we give people anything for posting their opinion of PostHog on software review sites?,correct,missing,grounded,appropriate,none,"None of the excerpts mention G2, review sites, or the $25 gift card review automation, so the assistant correctly sa..."
